In [ ]:
import os, time, glob
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch
import tensorflow as tf

torch.set_num_threads(os.cpu_count() or 4)
torch.set_grad_enabled(False)
tf.config.set_visible_devices([], "GPU")  # force CPU

def _find_one(patterns, must_contain):
    for pat in patterns:
        for hit in glob.glob(pat):
            p = Path(hit)
            if must_contain == "." or (p / must_contain).exists():
                return p
    return None

SR = 32_000
CLIP_SAMPLES = 5 * SR

COMP_DIR = _find_one(
    ["/kaggle/input/birdclef-2026",
     "/kaggle/input/competitions/birdclef-2026",
     "/kaggle/input/*birdclef*2026*"],
    must_contain="taxonomy.csv",
)
assert COMP_DIR is not None, "birdclef-2026 not attached"
TEST_DIR = COMP_DIR / "test_soundscapes"

# best_emb.pt (Model B head over Perch v2)
EMB_CANDS = sorted(glob.glob("/kaggle/input/**/best_emb.pt", recursive=True))
assert EMB_CANDS, "best_emb.pt not found - attach Model B Kaggle Model"
EMB_CKPT = Path(EMB_CANDS[0])

# Perch v2 SavedModel
PERCH = None
for hit in sorted(glob.glob("/kaggle/input/**/saved_model.pb", recursive=True)):
    s = str(Path(hit).parent).lower()
    if "bird" in s or "perch" in s or "vocaliz" in s:
        PERCH = Path(hit).parent; break
if PERCH is None:
    pbs = sorted(glob.glob("/kaggle/input/**/saved_model.pb", recursive=True))
    PERCH = Path(pbs[0]).parent if pbs else None
assert PERCH is not None, "Perch SavedModel not found - attach google/bird-vocalization-classifier"

OUT_PATH = Path("submission.csv")
tax = pd.read_csv(COMP_DIR / "taxonomy.csv")
CLASSES = tax["primary_label"].tolist()
NC = len(CLASSES)
print("data  :", COMP_DIR)
print("emb   :", EMB_CKPT)
print("perch :", PERCH)
print("classes:", NC)


In [ ]:
def load_audio(path):
    y, sr = sf.read(str(path), dtype="float32", always_2d=False)
    if y.ndim == 2:
        y = y.mean(axis=1)
    if sr != SR:
        y = librosa.resample(y, orig_sr=sr, target_sr=SR)
    return y.astype(np.float32, copy=False)

def segment_file(y, n=CLIP_SAMPLES):
    total = int(np.ceil(len(y) / n)) * n
    if len(y) < total:
        y = np.pad(y, (0, total - len(y)))
    return y.reshape(-1, n)


In [ ]:
# ---- Perch v2 embedder ---------------------------------------------------
_perch = tf.saved_model.load(str(PERCH))
_sig = _perch.signatures["serving_default"]
PERCH_IN_KEY = list(_sig.structured_input_signature[1].keys())[0]
_probe = _sig(**{PERCH_IN_KEY: tf.zeros((1, CLIP_SAMPLES), dtype=tf.float32)})
cands = []
for k, v in _probe.items():
    a = v.numpy() if hasattr(v, "numpy") else np.asarray(v)
    if a.ndim == 2 and a.shape[0] == 1 and a.shape[1] >= 256:
        cands.append((k, a.shape[1]))
cands.sort(key=lambda kv: (-(kv[1] == 1280), -kv[1]))
PERCH_OUT_KEY, EMB_DIM = cands[0]
print(f"perch in/out: {PERCH_IN_KEY} -> {PERCH_OUT_KEY}  dim={EMB_DIM}")

@tf.function(input_signature=[tf.TensorSpec(shape=[None, CLIP_SAMPLES], dtype=tf.float32)])
def perch_embed(batch):
    return _sig(**{PERCH_IN_KEY: batch})[PERCH_OUT_KEY]
_ = perch_embed(tf.zeros((2, CLIP_SAMPLES), dtype=tf.float32)).numpy()  # warmup

# ---- MLP head ------------------------------------------------------------
class MLPHead(torch.nn.Module):
    def __init__(self, d_in, n_classes, hidden=512, drop=0.3):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.LayerNorm(d_in),
            torch.nn.Linear(d_in, hidden), torch.nn.GELU(), torch.nn.Dropout(drop),
            torch.nn.Linear(hidden, hidden), torch.nn.GELU(), torch.nn.Dropout(drop),
            torch.nn.Linear(hidden, n_classes),
        )
    def forward(self, x): return self.net(x)

ck = torch.load(EMB_CKPT, map_location="cpu", weights_only=False)
cfg = ck.get("cfg", {})
hidden = int(cfg.get("hidden", 512))
drop   = float(cfg.get("drop", 0.3))
emb_d  = int(cfg.get("emb_dim", EMB_DIM))
assert emb_d == EMB_DIM, f"embed dim mismatch: ckpt={emb_d} vs perch={EMB_DIM}"
model = MLPHead(emb_d, NC, hidden=hidden, drop=drop)
model.load_state_dict(ck["model"]); model.eval()
print(f"Model B loaded  (val AUC at ckpt: {ck.get('auc', 'n/a')})")


In [ ]:
@torch.inference_mode()
def predict_segs(segs: np.ndarray, batch: int = 16) -> np.ndarray:
    out = np.empty((segs.shape[0], NC), dtype=np.float32)
    for i in range(0, segs.shape[0], batch):
        chunk = segs[i:i+batch].astype(np.float32, copy=False)
        emb = perch_embed(tf.constant(chunk)).numpy().astype(np.float32)
        logits = model(torch.from_numpy(emb))
        out[i:i+batch] = torch.sigmoid(logits).numpy().astype(np.float32)
    return out


In [ ]:
files = sorted(TEST_DIR.glob("*.ogg"))
print("test files:", len(files))

row_ids = []
all_probs = []

t0 = time.time()
with ThreadPoolExecutor(max_workers=min(4, os.cpu_count() or 2)) as pool:
    preloaded = {p: pool.submit(load_audio, p) for p in files[: min(4, len(files))]}
    for idx, p in enumerate(files):
        for q in files[idx + 1 : idx + 5]:
            if q not in preloaded:
                preloaded[q] = pool.submit(load_audio, q)
        y = preloaded.pop(p).result()

        segs = segment_file(y)
        probs = predict_segs(segs)

        # 3-tap temporal smoothing
        if probs.shape[0] >= 3:
            pad = np.pad(probs, ((1, 1), (0, 0)), mode="edge")
            probs = 0.2 * pad[:-2] + 0.6 * pad[1:-1] + 0.2 * pad[2:]
        # file-level prior
        probs = probs * (0.8 + 0.2 * probs.max(axis=0, keepdims=True))

        stem = p.stem
        for i in range(probs.shape[0]):
            end_sec = (i + 1) * 5
            row_ids.append(f"{stem}_{end_sec}")
        all_probs.append(probs)

        if (idx + 1) % 50 == 0:
            print(f"{idx+1}/{len(files)}   elapsed {time.time()-t0:.1f}s")

print("done. files:", len(files), "rows:", len(row_ids), "elapsed:", f"{time.time()-t0:.1f}s")


In [ ]:
if all_probs:
    arr = np.concatenate(all_probs, axis=0)
    sub = pd.DataFrame(arr, columns=CLASSES)
    sub.insert(0, "row_id", row_ids)
else:
    sub = pd.DataFrame(columns=["row_id"] + CLASSES)

sub.to_csv(OUT_PATH, index=False, float_format="%.5f")
print("wrote", OUT_PATH, sub.shape)
sub.head()
